# 01 — Preparar Chunks desde el Corpus JSONL

Genera chunks desde `corpus.jsonl` (formato LangChain: `page_content` + `metadata`).  
**Salida:** `chunks_<strategy>.jsonl` — listo para el Notebook 02 (indexación en Qdrant).

---

## Estrategias disponibles

Cambia únicamente `CHUNK_STRATEGY` en la celda de configuración:

| Tier | Estrategia | Clase LangChain | Estado |
|------|-----------|-----------------|--------|
| 1 | `fixed` | `TokenTextSplitter` — tamaño fijo en tokens | ✅ listo |
| 1 | `recursive` | `RecursiveCharacterTextSplitter` — split jerárquico | ✅ listo |
| 2 | `markdown` | `MarkdownHeaderTextSplitter` — todos los headings (#–######) | ✅ listo |
| 2 | `section` | `MarkdownHeaderTextSplitter` — solo heading `#` (secciones completas) | ✅ listo |
| 3 | `semantic` | `SemanticChunker` — por similitud semántica (requiere Ollama) | ✅ listo |
| 3 | `parent_child` | — | 🔜 próximamente |
| 3 | `sliding_window` | — | 🔜 próximamente |

In [31]:
%pip install langchain-text-splitters langchain-experimental tiktoken tqdm --quiet

Note: you may need to restart the kernel to use updated packages.


In [32]:
import json
import re
import statistics
from collections import Counter
from pathlib import Path

from tqdm.auto import tqdm
from langchain_text_splitters import (
    TokenTextSplitter,
    RecursiveCharacterTextSplitter,
    MarkdownHeaderTextSplitter,
)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  CONFIGURACIÓN — cambia solo esta celda (papermill la sobreescribe aquí)
# ══════════════════════════════════════════════════════════════════════════════

# Tier 1 token-based : "fixed_256" | "fixed_512" | "fixed_1024"
# Tier 1 word-based  : "recursive_200" | "recursive_400"
# Tier 2 estructura  : "markdown" | "section"
# Tier 3 semántico   : "semantic"
CHUNK_STRATEGY = "semantic"

# Paths absolutos — funcionan tanto en Jupyter como en papermill
CORPUS_PATH = "/home/coder/ia-testing/RAG1/data/corpus/corpus.jsonl"
OUTPUT_DIR  = "/home/coder/ia-testing/rageval/data/02_intermediate"

# Parámetros de tamaño por estrategia (size, overlap, unidad)
_PARAMS = {
    "fixed_256":     dict(chunk_size=256,   chunk_overlap=32,  unit="tokens"),
    "fixed_512":     dict(chunk_size=512,   chunk_overlap=64,  unit="tokens"),
    "fixed_1024":    dict(chunk_size=1024,  chunk_overlap=128, unit="tokens"),
    "recursive_200": dict(chunk_size=200,   chunk_overlap=25,  unit="words"),
    "recursive_400": dict(chunk_size=400,   chunk_overlap=50,  unit="words"),
    "fixed":         dict(chunk_size=512,   chunk_overlap=50,  unit="tokens"),
    "recursive":     dict(chunk_size=512,   chunk_overlap=50,  unit="words"),
    "markdown":      dict(chunk_size=512,   chunk_overlap=50,  unit="tokens"),
    "section":       dict(chunk_size=512,   chunk_overlap=50,  unit="tokens"),
    "semantic":      dict(chunk_size=0,     chunk_overlap=0,   unit="tokens"),
}

# ── Parámetros comunes ─────────────────────────────────────────────────────
MIN_WORDS      = 20
MD_CHUNK_WORDS = 450

# ── Ollama (solo para "semantic") ──────────────────────────────────────────
OLLAMA_URL  = "http://localhost:11434"
EMBED_MODEL = "nomic-embed-text"

In [ ]:
# Derivación — corre DESPUÉS del override de papermill, no modificar
_p            = _PARAMS[CHUNK_STRATEGY]
CHUNK_SIZE    = _p["chunk_size"]
CHUNK_OVERLAP = _p["chunk_overlap"]
OUTPUT_PATH   = f"{OUTPUT_DIR}/chunks_{CHUNK_STRATEGY}.jsonl"

print(f"Estrategia : {CHUNK_STRATEGY}")
print(f"Size       : {CHUNK_SIZE}  ({_p['unit']})")
print(f"Overlap    : {CHUNK_OVERLAP}  ({_p['unit']})")
print(f"Corpus     : {CORPUS_PATH}")
print(f"Salida     : {OUTPUT_PATH}")

## 1. Cargar corpus

In [34]:
docs = []
with open(CORPUS_PATH, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            docs.append(json.loads(line))

print(f"Documentos cargados: {len(docs)}")
print(f"Estrategia activa  : {CHUNK_STRATEGY} (size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP})")
print()
tipos = Counter(d["metadata"].get("doc_type", "desconocido") for d in docs)
print("Distribución por tipo:")
for tipo, n in sorted(tipos.items(), key=lambda x: -x[1]):
    print(f"  {tipo:30s} {n}")

Documentos cargados: 124
Estrategia activa  : fixed (size=512, overlap=50)

Distribución por tipo:
  instruccion_tecnica            67
  proceso                        34
  politica                       12
  manual                         4
  reglamento                     4
  documento                      3


## 2. Utilidades comunes

In [35]:
def get_doc_id(metadata: dict) -> str:
    doc_code = metadata.get("doc_code")
    if doc_code:
        return str(doc_code)
    file_name = metadata.get("file_name", "unknown")
    return re.sub(r'\.(odt|docx)?\.?md$', '', file_name, flags=re.IGNORECASE)


def detect_semantic_flags(text: str) -> dict[str, bool]:
    """Flags semánticos útiles para filtrado y análisis RAG."""
    has_table = bool(
        re.search(r'^\|.+\|', text, re.MULTILINE)
        or re.search(r'^\+[-+]{3,}\+', text, re.MULTILINE)
    )
    has_steps = len(re.findall(r'^\s*\d+[\.):]\s+\S', text, re.MULTILINE)) >= 3
    has_roles = bool(re.search(
        r'\b(responsable|perfil|rol|cargo|supervisor|director|jefe|gerente)\b',
        text, re.IGNORECASE
    ))
    has_definitions = bool(re.search(
        r'(se entiende por|se define|significa que|es decir)',
        text, re.IGNORECASE
    ))
    has_requirements = bool(re.search(
        r'\b(obligatorio|deber[aá]|debe|prohibido|es necesario|se requiere|han de)\b',
        text, re.IGNORECASE
    ))
    has_exceptions = bool(re.search(
        r'(excepto|a excepci[oó]n de|salvo que|salvo|no aplica)',
        text, re.IGNORECASE
    ))
    return {
        "has_table": has_table,
        "has_steps": has_steps,
        "has_roles": has_roles,
        "has_definitions": has_definitions,
        "has_requirements": has_requirements,
        "has_exceptions": has_exceptions,
    }


def _get(meta: dict, key: str, cast=None, default=None):
    """Safe metadata accessor with optional type cast."""
    val = meta.get(key)
    if val is None:
        return default
    return cast(val) if cast else val


_W = 62  # ancho de línea para display_chunk


def display_chunk(chunk: dict, idx: int | None = None) -> None:
    """Imprime todos los campos de metadata y el texto completo del chunk."""
    m = chunk["metadata"]
    label = f" CHUNK {idx} " if idx is not None else " CHUNK "

    def sep(title=""):
        if title:
            print(f"\n── {title} " + "─" * (_W - len(title) - 4))
        else:
            print("═" * _W)

    def row(key, val):
        print(f"  {key:<22}: {val}")

    sep(); print(label.center(_W, "═")); sep()

    sep("Identidad")
    row("chunk_id",   m.get("chunk_id", ""))
    row("doc_id",     m.get("doc_id", ""))
    row("doc_code",   m.get("doc_code", ""))
    row("file_name",  m.get("file_name", ""))

    sep("Estrategia")
    row("chunk_strategy", f"{m.get('chunk_strategy', '')}  (size={m.get('chunk_size', '')}, overlap={m.get('chunk_overlap', '')})")

    sep("Documento")
    for key in ("doc_type", "doc_title", "doc_version", "doc_date", "process_area",
                "doc_process_family", "doc_total_words", "doc_in_degree", "doc_out_degree",
                "parent_process_file", "parent_process_title", "sibling_its"):
        row(key, m.get(key, ""))

    sep("Posición")
    row("chunk_index", f"{m.get('chunk_index', '')} / {m.get('chunk_total', '')}")
    row("word_count",  m.get("word_count", 0))
    row("char_count",  m.get("char_count", 0))

    sep("Sección (Tier 2)")
    row("section_title", m.get("section_title") or "(tier 1 — sin heading)")
    row("section_level", m.get("section_level") or "(tier 1 — sin heading)")
    row("breadcrumb",    m.get("breadcrumb")    or "(tier 1 — sin breadcrumb)")

    sep("Flags semánticos")
    flags = [(k, v) for k, v in m.items() if k.startswith("has_")]
    for i in range(0, len(flags), 2):
        left  = f"{flags[i][0]:<20}: {str(flags[i][1]):<6}"
        right = f"{flags[i+1][0]:<20}: {flags[i+1][1]}" if i + 1 < len(flags) else ""
        print(f"  {left}  {right}")

    sep("Texto completo")
    print()
    print(chunk["text"])
    print()
    sep()


# Tipo de retorno que todas las estrategias deben devolver:
# list[dict] con campos: text, section_title, breadcrumb, section_level
# (section_title/breadcrumb/section_level vacíos en Tier 1)
ChunkResult = list[dict]

## 3. Estrategias de chunking

In [36]:
# ══════════════════════════════════════════════════════════════════
#  TIER 1 — Fixed-size  (TokenTextSplitter, medido en tokens)
# ══════════════════════════════════════════════════════════════════

def strategy_fixed(text: str) -> ChunkResult:
    splitter = TokenTextSplitter(
        encoding_name="cl100k_base",  # tokenizer GPT-4, buena aproximación general
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
    )
    return [
        {"text": t, "section_title": "", "breadcrumb": "", "section_level": ""}
        for t in splitter.split_text(text)
        if len(t.split()) >= MIN_WORDS
    ]

In [37]:
# ══════════════════════════════════════════════════════════════════
#  TIER 1 — Recursive  (RecursiveCharacterTextSplitter, medido en palabras)
# ══════════════════════════════════════════════════════════════════

def strategy_recursive(text: str) -> ChunkResult:
    splitter = RecursiveCharacterTextSplitter(
        # separadores por defecto: ["\n\n", "\n", " ", ""]
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        length_function=lambda x: len(x.split()),   # medido en PALABRAS
    )
    return [
        {"text": t, "section_title": "", "breadcrumb": "", "section_level": ""}
        for t in splitter.split_text(text)
        if len(t.split()) >= MIN_WORDS
    ]

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  TIER 2 — Markdown / Header-based
#  MarkdownHeaderTextSplitter preserva la jerarquía de headings
#  como metadata → section_title, breadcrumb, section_level
# ══════════════════════════════════════════════════════════════════

_MD_HEADERS = [
    ("#", "h1"), ("##", "h2"), ("###", "h3"),
    ("####", "h4"), ("#####", "h5"), ("######", "h6"),
]

def strategy_markdown(text: str) -> ChunkResult:
    md_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=_MD_HEADERS,
        strip_headers=False,   # incluir heading en el texto → mejor embedding
    )
    size_splitter = RecursiveCharacterTextSplitter(
        chunk_size=MD_CHUNK_WORDS,
        chunk_overlap=CHUNK_OVERLAP,
        length_function=lambda x: len(x.split()),
    )

    result: ChunkResult = []
    for doc in md_splitter.split_text(text):
        # doc.metadata = {"h1": "Título", "h2": "Sección", ...}
        headers = {k: v for k, v in doc.metadata.items() if v}
        breadcrumb    = " > ".join(headers[k] for k in sorted(headers) if headers[k])
        section_title = list(headers.values())[-1] if headers else ""
        section_level = list(headers.keys())[-1]   if headers else ""
        parent_text   = doc.page_content  # sección completa antes del split secundario

        pieces = (
            size_splitter.split_text(doc.page_content)
            if len(doc.page_content.split()) > MD_CHUNK_WORDS
            else [doc.page_content]
        )
        for t in pieces:
            if len(t.split()) >= MIN_WORDS:
                result.append({
                    "text":          t,
                    "parent_text":   parent_text,
                    "section_title": section_title,
                    "breadcrumb":    breadcrumb,
                    "section_level": section_level,
                })
    return result

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  TIER 2 — Section-based
#  Solo divide en heading `#` (h1), preservando sub-secciones
#  completas dentro del mismo chunk. Chunks más grandes y cohesivos
#  que "markdown", que divide en todos los niveles de heading.
# ══════════════════════════════════════════════════════════════════

_SECTION_HEADERS = [("#", "h1")]

def strategy_section(text: str) -> ChunkResult:
    md_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=_SECTION_HEADERS,
        strip_headers=False,
    )
    size_splitter = RecursiveCharacterTextSplitter(
        chunk_size=MD_CHUNK_WORDS,
        chunk_overlap=CHUNK_OVERLAP,
        length_function=lambda x: len(x.split()),
    )

    result: ChunkResult = []
    for doc in md_splitter.split_text(text):
        headers = {k: v for k, v in doc.metadata.items() if v}
        section_title = list(headers.values())[-1] if headers else ""
        section_level = list(headers.keys())[-1]   if headers else ""
        breadcrumb    = section_title
        parent_text   = doc.page_content  # sección h1 completa antes del split

        pieces = (
            size_splitter.split_text(doc.page_content)
            if len(doc.page_content.split()) > MD_CHUNK_WORDS
            else [doc.page_content]
        )
        for t in pieces:
            if len(t.split()) >= MIN_WORDS:
                result.append({
                    "text":          t,
                    "parent_text":   parent_text,
                    "section_title": section_title,
                    "breadcrumb":    breadcrumb,
                    "section_level": section_level,
                })
    return result


# ══════════════════════════════════════════════════════════════════
#  TIER 3 — Semantic  (SemanticChunker, requiere langchain-experimental)
#  Divide el texto en función de la similitud semántica entre frases,
#  usando embeddings de Ollama. Las fronteras se colocan donde hay
#  un salto semántico significativo (percentil 95 por defecto).
#  ⚠️  Lento: hace llamadas al modelo de embeddings durante el chunking.
# ══════════════════════════════════════════════════════════════════

import re as _re
import requests as _requests
from langchain_core.embeddings import Embeddings as _Embeddings

# Límite conservador para tablas markdown: muchos `|` + espacios tokenizan
# ineficientemente (~1-2 chars/token), así que 1500 chars ≈ 750-1500 tokens,
# bien dentro del límite de nomic-embed-text en Ollama (~2048 por defecto).
# Para detectar fronteras semánticas no necesitamos más contexto que este.
_EMBED_MAX_CHARS = 1500


def _sanitize(text: str) -> str:
    """Elimina caracteres de control (null bytes, etc.) que Ollama rechaza con 400."""
    return _re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', text)


class _OllamaDirectEmbeddings(_Embeddings):
    """Wrapper mínimo sobre /api/embed de Ollama para SemanticChunker."""
    _dim: int | None = None

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return [self._embed(t) for t in texts]

    def embed_query(self, text: str) -> list[float]:
        return self._embed(text)

    def _embed(self, text: str) -> list[float]:
        clean = _sanitize(text).strip()
        if not clean:
            return [0.0] * (self._dim or 768)
        resp = _requests.post(
            f"{OLLAMA_URL}/api/embed",
            json={"model": EMBED_MODEL, "input": clean[:_EMBED_MAX_CHARS]},
            timeout=120,
        )
        if not resp.ok:
            print(f"[WARN] embed {resp.status_code} — {resp.text[:200]!r} | texto[:80]: {clean[:80]!r}")
            return [0.0] * (self._dim or 768)
        vec = resp.json()["embeddings"][0]
        if self._dim is None:
            self.__class__._dim = len(vec)
        return vec


def strategy_semantic(text: str) -> ChunkResult:
    from langchain_experimental.text_splitter import SemanticChunker

    splitter = SemanticChunker(
        _OllamaDirectEmbeddings(),
        breakpoint_threshold_type="percentile",
        breakpoint_threshold_amount=95,
    )
    return [
        {"text": t, "section_title": "", "breadcrumb": "", "section_level": ""}
        for t in splitter.split_text(text)
        if len(t.split()) >= MIN_WORDS
    ]


# ══════════════════════════════════════════════════════════════════
#  TIER 3 — Parent-child           (próximamente)
#  TIER 3 — Sliding window         (próximamente)
# ══════════════════════════════════════════════════════════════════

## 4. Dispatcher y construcción de metadatos

In [ ]:
# Registro de estrategias — añade aquí las nuevas
STRATEGIES = {
    # Tier 1 — token-based
    "fixed_256":     strategy_fixed,
    "fixed_512":     strategy_fixed,
    "fixed_1024":    strategy_fixed,
    # Tier 1 — word-based
    "recursive_200": strategy_recursive,
    "recursive_400": strategy_recursive,
    # legados
    "fixed":         strategy_fixed,
    "recursive":     strategy_recursive,
    # Tier 2
    "markdown":      strategy_markdown,
    "section":       strategy_section,
    # Tier 3
    "semantic":      strategy_semantic,
}

if CHUNK_STRATEGY not in STRATEGIES:
    raise ValueError(
        f"Estrategia '{CHUNK_STRATEGY}' no reconocida. "
        f"Disponibles: {list(STRATEGIES.keys())}"
    )

_strategy_fn = STRATEGIES[CHUNK_STRATEGY]
print(f"Estrategia : {CHUNK_STRATEGY}")
print(f"Función    : {_strategy_fn.__name__}  (size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP})")

In [ ]:
def chunk_document(page_content: str, doc_meta: dict) -> list[dict]:
    """Aplica la estrategia activa y devuelve lista de chunks con metadatos completos."""
    doc_id = get_doc_id(doc_meta)
    raw: ChunkResult = _strategy_fn(page_content)
    chunk_total = len(raw)

    result = []
    for idx, c in enumerate(raw):
        flags = detect_semantic_flags(c["text"])
        result.append({
            "text": c["text"],
            "metadata": {
                # ── Identidad ─────────────────────────────────────────────
                "chunk_id":       f"{doc_id}_chunk_{idx:03d}",
                "doc_id":         doc_id,
                "doc_code":       doc_meta.get("doc_code"),
                "file_name":      doc_meta.get("file_name"),
                # ── Estrategia (trazabilidad) ──────────────────────────────
                "chunk_strategy": CHUNK_STRATEGY,
                "chunk_size":     CHUNK_SIZE,
                "chunk_overlap":  CHUNK_OVERLAP,
                # ── Metadatos del documento ────────────────────────────────
                "doc_type":             doc_meta.get("doc_type"),
                "doc_title":            doc_meta.get("doc_title"),
                "doc_version":          _get(doc_meta, "doc_version", str),
                "doc_date":             doc_meta.get("doc_date"),
                "process_area":         doc_meta.get("process_area"),
                "doc_process_family":   doc_meta.get("doc_process_family"),
                "doc_total_words":      _get(doc_meta, "doc_total_words", int, default=0),
                "doc_in_degree":        _get(doc_meta, "doc_in_degree",   int, default=0),
                "doc_out_degree":       _get(doc_meta, "doc_out_degree",  int, default=0),
                "parent_process_file":  doc_meta.get("parent_process_file"),
                "parent_process_title": doc_meta.get("parent_process_title"),
                "sibling_its":          doc_meta.get("sibling_its", ""),
                # ── Posición del chunk ─────────────────────────────────────
                "chunk_index": idx,
                "chunk_total": chunk_total,
                "word_count":  len(c["text"].split()),
                "char_count":  len(c["text"]),
                # ── Contexto de sección (vacío en Tier 1) ─────────────────
                "section_title": c["section_title"],
                "breadcrumb":    c["breadcrumb"],
                "section_level": c["section_level"],
                "parent_text":   c.get("parent_text", ""),
                # ── Flags semánticos ───────────────────────────────────────
                **flags,
            },
        })

    return result

## 5. Ejecutar chunking

In [42]:
all_chunks = []

for doc in tqdm(docs, desc=f"Chunking [{CHUNK_STRATEGY}]"):
    chunks = chunk_document(doc["page_content"], doc["metadata"])
    all_chunks.extend(chunks)

print(f"\nTotal chunks generados: {len(all_chunks)}")

Chunking [fixed]:   0%|          | 0/124 [00:00<?, ?it/s]


Total chunks generados: 4796


## 6. Inspección de un chunk de ejemplo

In [43]:
display_chunk(all_chunks[5], idx=5)

══════════════════════════════════════════════════════════════
══════════════════════════ CHUNK 5 ═══════════════════════════
══════════════════════════════════════════════════════════════

── Identidad ─────────────────────────────────────────────────
  chunk_id              : IT_01_01_chunk_002
  doc_id                : IT_01_01
  doc_code              : IT_01_01
  file_name             : IT_01_01_Control_de_informacion_documentada.odt.md

── Estrategia ────────────────────────────────────────────────
  chunk_strategy        : fixed  (size=512, overlap=50)

── Documento ─────────────────────────────────────────────────
  doc_type              : instruccion_tecnica
  doc_title             : IT_01_01 Control de información documentada
  doc_version           : 17.0
  doc_date              : 2025-02-11
  process_area          : estrategicos
  doc_process_family    : procesos_estrategicos
  doc_total_words       : 3332
  doc_in_degree         : 10
  doc_out_degree        : 9
  parent_pro

## 7. Estadísticas

In [44]:
word_counts = [c["metadata"]["word_count"] for c in all_chunks]

print(f"Estrategia     : {CHUNK_STRATEGY}")
print(f"Chunks totales : {len(all_chunks)}")
print(f"Palabras/chunk — min: {min(word_counts)}, max: {max(word_counts)}, "
      f"media: {statistics.mean(word_counts):.0f}, mediana: {statistics.median(word_counts):.0f}")
print()

flags_keys = ["has_table", "has_steps", "has_roles",
              "has_definitions", "has_requirements", "has_exceptions"]
print("Flags semánticos:")
for flag in flags_keys:
    count = sum(1 for c in all_chunks if c["metadata"].get(flag))
    pct = 100 * count / len(all_chunks)
    print(f"  {flag:20s} {count:4d}  ({pct:.1f}%)")

print()
areas = Counter(c["metadata"].get("process_area", "") for c in all_chunks)
print("Chunks por process_area:")
for area, n in sorted(areas.items(), key=lambda x: -x[1]):
    print(f"  {area or '(sin área)':30s} {n}")

Estrategia     : fixed
Chunks totales : 4796
Palabras/chunk — min: 30, max: 361, media: 296, mediana: 301

Flags semánticos:
  has_table            2648  (55.2%)
  has_steps             244  (5.1%)
  has_roles             984  (20.5%)
  has_definitions       202  (4.2%)
  has_requirements      910  (19.0%)
  has_exceptions        231  (4.8%)

Chunks por process_area:
  soporte_admin                  2542
  estrategicos                   841
  principales                    702
  soporte_tecnico                673
  (sin área)                     38


## 8. Guardar `chunks_<strategy>.jsonl`

In [45]:
out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w", encoding="utf-8") as f:
    for chunk in all_chunks:
        f.write(json.dumps(chunk, ensure_ascii=False) + "\n")

size_kb = out_path.stat().st_size / 1024
print(f"Guardado : {out_path}")
print(f"Tamaño   : {size_kb:.1f} KB")
print(f"Líneas   : {len(all_chunks)}")

Guardado : ../../rageval/data/02_intermediate/chunks_fixed.jsonl
Tamaño   : 21542.3 KB
Líneas   : 4796
